# Carrot tuning system: initial model

This notebook is a **high-level, executable mock** of the system we are building. It does not call Azure, Foundry, an LLM, or Kaggriculture. It does not edit policy source.

Its purpose is to make the intended control flow concrete before we build the durable components: a deterministic inner experiment loop, an evidence packet, an outer evals/reasoning loop, and a human approval boundary.

The completed carrot experiments are used only as illustrative input data. The notebook’s evaluator is intentionally mocked.

## 1. The idea in one picture

```text
Human approves a bounded experiment specification
                 |
                 v
Inner loop (local and deterministic)
  baseline -> checkpoint replay/suffix -> clean audit -> evidence packet
                 |
                 v
Outer loop (evals + reasoning)
  validate evidence -> explain result -> propose next bounded test
                 |
                 v
Human approves the next envelope or applies a reviewed source change
```

The evaluator is the score authority because it runs Kaggriculture locally. The outer loop is an interpretation and experiment-design authority, not a source-editing authority.

## 2. Guardrails encoded as data

A useful first design choice is to make the approved envelope explicit. The system can automatically run values inside this specification, but it cannot silently add values, change the target source, or promote a value into policy.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Literal


@dataclass(frozen=True)
class Suite:
    """Matched controls for every candidate in one experiment."""

    opponent_id: str
    seeds: tuple[int, ...]
    seats: tuple[int, ...] = (0, 1)
    steps: int = 720


@dataclass(frozen=True)
class ExperimentSpec:
    """A human-approved, bounded inner-loop instruction."""

    experiment_id: str
    decision_graph_id: str
    parameter_name: str
    baseline_value: float
    candidate_values: tuple[float, ...]
    suite: Suite
    purpose: str
    requires_clean_audit: bool = True


coarse_seed_bracket = ExperimentSpec(
    experiment_id="carrot-seed-opportunity-coarse-v1",
    decision_graph_id="carrot_investment",
    parameter_name="seed_opportunity_value",
    baseline_value=20.0,
    candidate_values=(10.0, 40.0, 60.0, 80.0),
    suite=Suite(
        opponent_id="target.agents.carrot.shared.carrot_conveyor_agent",
        seeds=tuple(range(42, 72)),
    ),
    purpose="Measure the direction and threshold behavior of preserving a carrot seed.",
)

coarse_seed_bracket

## 3. Inner-loop output: an evidence packet

The real inner loop will invoke the local checkpoint/resume evaluator. A candidate follows baseline observations until its first different action; it reuses the baseline when there is no divergence, otherwise restores the exact pre-decision state and simulates the suffix. A clean full rerun audits every result.

For this notebook, the mock evaluator returns a frozen snapshot of the completed coarse `seed_opportunity_value` run. This is deliberately not a simulation substitute. It only demonstrates the data contract the real evaluator should produce.

In [ ]:
@dataclass(frozen=True)
class CandidateEvidence:
    value: float
    mean_reward: float
    mean_paired_delta: float
    diverged_matches: int
    total_matches: int
    clean_audits_passed: int
    audit_status: Literal["passed", "failed"]


@dataclass(frozen=True)
class EvidencePacket:
    experiment_id: str
    decision_graph_id: str
    target_revision: str
    suite: Suite
    baseline_mean_reward: float
    candidates: tuple[CandidateEvidence, ...]
    evaluator_status: Literal["complete", "audit_failed"]
    notes: tuple[str, ...] = ()


# Frozen illustrative evidence from the already-completed local coarse screen.
# The real implementation will build this from JSON result files.
HISTORICAL_SEED_RESULTS = {
    10.0: (6469.02, 0.00, 0),
    40.0: (6416.65, -52.37, 24),
    60.0: (5631.28, -837.73, 48),
    80.0: (4845.73, -1623.28, 60),
}


def mock_inner_loop(spec: ExperimentSpec) -> EvidencePacket:
    """Mock only: returns fixture data and never runs a game."""
    expected_matches = len(spec.suite.seeds) * len(spec.suite.seats)
    candidates = tuple(
        CandidateEvidence(
            value=value,
            mean_reward=HISTORICAL_SEED_RESULTS[value][0],
            mean_paired_delta=HISTORICAL_SEED_RESULTS[value][1],
            diverged_matches=HISTORICAL_SEED_RESULTS[value][2],
            total_matches=expected_matches,
            clean_audits_passed=expected_matches,
            audit_status="passed",
        )
        for value in spec.candidate_values
    )
    return EvidencePacket(
        experiment_id=spec.experiment_id,
        decision_graph_id=spec.decision_graph_id,
        target_revision="6ace5fed9018fef45642b41ea11d0bd741a508a4",
        suite=spec.suite,
        baseline_mean_reward=6469.02,
        candidates=candidates,
        evaluator_status="complete",
        notes=("Mock packet populated from completed local evidence.",),
    )


packet = mock_inner_loop(coarse_seed_bracket)
packet

## 4. Inspect the deterministic evidence before reasoning

This is the critical separation: the evaluation layer creates facts about the stated experiment; the reasoning layer is not allowed to reinterpret an audit failure into success.

In [ ]:
for row in packet.candidates:
    print(
        f"value={row.value:>5.0f} | reward={row.mean_reward:>7.2f} "
        f"| delta={row.mean_paired_delta:>8.2f} "
        f"| divergence={row.diverged_matches}/{row.total_matches} "
        f"| audit={row.audit_status}"
    )

## 5. Outer evals loop: classify evidence, do not make policy edits

The first outer-loop component can be deterministic too. It checks that audits passed, identifies whether a candidate was inert or harmful on this suite, and emits constraints for the reasoning step. A future Foundry evaluator can implement richer review, but it should preserve these non-negotiable checks.

In [ ]:
@dataclass(frozen=True)
class EvaluationAssessment:
    status: Literal["evidence_valid", "audit_failed"]
    findings: tuple[str, ...]
    prohibited_next_values: tuple[float, ...]


def evaluate_packet(packet: EvidencePacket) -> EvaluationAssessment:
    if packet.evaluator_status != "complete" or any(
        row.audit_status != "passed" for row in packet.candidates
    ):
        return EvaluationAssessment(
            status="audit_failed",
            findings=("Do not interpret or extend this experiment until audits are repaired.",),
            prohibited_next_values=(),
        )

    inert = [row.value for row in packet.candidates if row.diverged_matches == 0]
    harmful = [row.value for row in packet.candidates if row.mean_paired_delta < 0]
    return EvaluationAssessment(
        status="evidence_valid",
        findings=(
            f"Inert values on this suite: {inert}.",
            f"Values with negative mean paired reward: {harmful}.",
            "The current evidence does not support increasing seed opportunity above 20.",
        ),
        prohibited_next_values=tuple(harmful),
    )


assessment = evaluate_packet(packet)
assessment

## 6. Reasoning loop: propose the next bounded uncertainty to resolve

This mock represents the job of a future reasoning agent. It receives the evidence packet and evaluation assessment, explains a proposed next test, and returns a bounded specification. It cannot write Kaggriculture source or expand its own candidate envelope.

Here, the correct next question is not “try an even higher value.” The observed bracket already shows higher static seed opportunity is harmful for the conveyor suite. The useful next question is whether this value should be supplied by a competing decision graph in different farm contexts.

In [ ]:
@dataclass(frozen=True)
class ReasoningProposal:
    hypothesis: str
    explanation: str
    next_experiment: ExperimentSpec
    requires_human_approval: bool = True


def mock_reasoning_loop(packet: EvidencePacket, assessment: EvaluationAssessment) -> ReasoningProposal:
    assert assessment.status == "evidence_valid", "Reasoning must stop on an audit failure."
    next_spec = ExperimentSpec(
        experiment_id="carrot-seed-opportunity-context-smoke-v1",
        decision_graph_id="carrot_investment",
        parameter_name="seed_opportunity_value",
        baseline_value=20.0,
        candidate_values=(10.0, 20.0, 40.0),
        suite=Suite(
            opponent_id="future.deterministic.competing-opportunity-scenario",
            seeds=tuple(range(72, 82)),
        ),
        purpose="Test whether a different deterministic opportunity regime changes the seed-option threshold.",
    )
    return ReasoningProposal(
        hypothesis=(
            "A static seed opportunity of 20 is supported against the conveyor, "
            "but another decision graph may create states where retaining the option has greater value."
        ),
        explanation=(
            "The completed bracket rules out higher static values for the observed suite, "
            "so the next test changes scenario coverage rather than blindly extending the same range."
        ),
        next_experiment=next_spec,
    )


proposal = mock_reasoning_loop(packet, assessment)
print(proposal.hypothesis)
print()
print(proposal.explanation)
print()
print(proposal.next_experiment)

## 7. The future integration seam: contextual seed opportunity

The carrot graph should continue to estimate its own carrot return. A higher-level planner can eventually provide the value of preserving the seed/tile/action capacity. The following is only a transparent sketch of that interface—not an economic model and not a policy change.

In [ ]:
@dataclass(frozen=True)
class FarmContext:
    alternative_crop_value: float
    tile_scarcity_value: float
    farmer_time_pressure: float


def mock_opportunity_value(context: FarmContext) -> float:
    """
    Future planner-to-carrot-graph interface.

    The weights are placeholders. A real graph must provide provenance and
    be calibrated by the same local evaluator—not trusted because it sounds reasonable.
    """
    return (
        context.alternative_crop_value
        + context.tile_scarcity_value
        + context.farmer_time_pressure
    )


quiet_farm = FarmContext(alternative_crop_value=8, tile_scarcity_value=4, farmer_time_pressure=3)
contested_farm = FarmContext(alternative_crop_value=18, tile_scarcity_value=12, farmer_time_pressure=10)

for name, context in [("quiet farm", quiet_farm), ("contested farm", contested_farm)]:
    print(f"{name:15}: proposed opportunity value = {mock_opportunity_value(context):.1f}")

## 8. What this notebook deliberately does *not* do

- It does not claim that the mock contextual formula is valid.
- It does not call the real evaluator or run a simulator.
- It does not provision Foundry or Azure resources.
- It does not use a model call.
- It does not modify `CarrotParameters` in Kaggriculture source.
- It does not allow the reasoning layer to override failed audits or write a policy change.

## 9. Concrete next implementation step

Replace `mock_inner_loop` with a local adapter that validates a versioned experiment specification and calls the existing checkpoint evaluator. Persist an `EvidencePacket` JSON file with the schema sketched here. Keep `evaluate_packet` deterministic at first. Only then replace the mock reasoning function with a Foundry-facing agent adapter, behind the same evidence-packet and human-approval boundaries.